In [1]:
import os
import sys
import urllib.request

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, RepeatVector, TimeDistributed


In [2]:
parser = (lambda x:datetime.datetime.strptime(x, '%Y.%m.%d')) 
df = pd.read_csv('sp_beaches_update.csv', parse_dates=['Date'])
df=df.loc[~df['Enterococcus'].isnull()]
#remover a praia do Leste, da cidade de iguape, pois esta praia sumiu por erosão em 2012
#remover a Lagoa Prumirim, da cidade de Ubatuba, pois esta praia possui somente 3 medições
df = df.loc[df['Beach']!='DO LESTE'].loc[df['Beach']!='LAGOA PRUMIRIM']
df['praia'] = df[['City', 'Beach']].apply(lambda x: ' - '.join(x), axis=1)
print(f'Numero de praias: {len(df.praia.unique())}')
df = df.sort_values(by=['praia'])
df = df.sort_values(by=['Date'])
df = df[['Date','praia','Enterococcus']]
df.shape

Numero de praias: 174


(69016, 3)

In [3]:
df_aux=df.pivot(index='praia', columns='Date', values='Enterococcus' )
df_aux.fillna(df_aux.mean(), inplace=True)
df_aux=df_aux.astype('int32')
df_aux

Date,2012-01-03,2012-01-08,2012-01-15,2012-01-22,2012-01-29,2012-02-05,2012-02-12,2012-02-19,2012-02-26,2012-03-04,...,2020-07-27,2020-08-03,2020-08-10,2020-08-17,2020-08-24,2020-08-31,2020-09-07,2020-09-14,2020-09-21,2020-09-28
praia,,,,,,,,,,,,,,,,,,,,,
BERTIOGA - BORACÉIA - COL. MARISTA,8,22,17,8,2,1,68,32,1,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - BORACÉIA - SUL,16,1,1,15,7,2,18,45,4,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - COLÔNIA DO SESC,128,42,18,31,11,19,252,36,1,2,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - INDAIÁ,17,85,1,10,108,6,7,29,15,1,...,15,20,18,48,33,35,39,36,287,91
BERTIOGA - ENSEADA - R. RAFAEL COSTABILI,204,34,19,33,10,25,288,57,42,2,...,3,43,3,30,30,8,2,2,152,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UBATUBA - SUNUNGA,5,6,34,9,3,5,9,2,3,1,...,15,20,18,48,33,35,39,36,287,91
UBATUBA - TENÓRIO,9,96,42,14,7,17,5,1,12,5,...,9,1,3,14,2,1,12,1,86,4
UBATUBA - TONINHAS,1,128,21,11,13,2,13,21,28,2,...,36,9,3,7,9,18,10,2,188,41


In [4]:
test_size=20
treino = df_aux.iloc[:,-104-test_size:-test_size].copy().T
treino = treino.values.reshape((1,treino.shape[0],treino.shape[1]))
teste = df_aux.iloc[:,-test_size:].copy().T
teste = teste.values.reshape((1,teste.shape[0],teste.shape[1]))
valida = df_aux.iloc[:,-104:].copy().T
valida = valida.values.reshape((1,valida.shape[0],valida.shape[1]))
print(treino.shape)
print(teste.shape)
print(valida.shape)


(1, 104, 174)
(1, 20, 174)
(1, 104, 174)


In [5]:
treino

array([[[ 12,   3,   4, ..., 124,   2,   2],
        [  2,   1,  38, ...,  96,  21,  16],
        [ 67,  71,  78, ..., 104,   9,  42],
        ...,
        [  2,   1,  14, ..., 104,   1,   7],
        [  1,  12,  19, ...,   3,   2,   2],
        [ 15,  42,  77, ...,   1,   3,   6]]])

In [6]:
# define model
model = Sequential()
model.add(LSTM(200, activation='relu', input_shape=(treino.shape[1],treino.shape[2])))
model.add(RepeatVector(test_size))
model.add(LSTM(200, activation='relu', return_sequences=True))
model.add(TimeDistributed(Dense(treino.shape[2])))
model.compile(optimizer='adam', loss='mean_squared_logarithmic_error')

In [7]:
# fit model
with tf.device('/GPU:0'):
    R = model.fit(treino, teste, epochs=300, verbose=1)
R

Train on 1 samples
Epoch 1/300
1/1 [==============================] - 5s 5s/sample - loss: 9.8929
Epoch 2/300
1/1 [==============================] - 0s 71ms/sample - loss: 18.8451
Epoch 3/300
1/1 [==============================] - 0s 67ms/sample - loss: 15.9236
Epoch 4/300
1/1 [==============================] - 0s 67ms/sample - loss: 21.7682
Epoch 5/300
1/1 [==============================] - 0s 69ms/sample - loss: 13.4407
Epoch 6/300
1/1 [==============================] - 0s 73ms/sample - loss: 13.4407
Epoch 7/300
1/1 [==============================] - 0s 65ms/sample - loss: 13.4407
Epoch 8/300
1/1 [==============================] - 0s 66ms/sample - loss: 13.4407
Epoch 9/300
1/1 [==============================] - 0s 68ms/sample - loss: 13.4407
Epoch 10/300
1/1 [==============================] - 0s 70ms/sample - loss: 13.4407
Epoch 11/300
1/1 [==============================] - 0s 71ms/sample - loss: 13.4407
Epoch 12/300
1/1 [==============================] - 0s 74ms/sample - loss: 13.44

Epoch 100/300
1/1 [==============================] - 0s 66ms/sample - loss: 13.4407
Epoch 101/300
1/1 [==============================] - 0s 63ms/sample - loss: 13.4407
Epoch 102/300
1/1 [==============================] - 0s 69ms/sample - loss: 13.4407
Epoch 103/300
1/1 [==============================] - 0s 71ms/sample - loss: 13.4407
Epoch 104/300
1/1 [==============================] - 0s 69ms/sample - loss: 13.4407
Epoch 105/300
1/1 [==============================] - 0s 67ms/sample - loss: 13.4407
Epoch 106/300
1/1 [==============================] - 0s 75ms/sample - loss: 13.4407
Epoch 107/300
1/1 [==============================] - 0s 77ms/sample - loss: 13.4407
Epoch 108/300
1/1 [==============================] - 0s 88ms/sample - loss: 13.4407
Epoch 109/300
1/1 [==============================] - 0s 75ms/sample - loss: 13.4407
Epoch 110/300
1/1 [==============================] - 0s 73ms/sample - loss: 13.4407
Epoch 111/300
1/1 [==============================] - 0s 83ms/sample - loss: 

Epoch 198/300
1/1 [==============================] - 0s 76ms/sample - loss: 13.4407
Epoch 199/300
1/1 [==============================] - 0s 68ms/sample - loss: 13.4407
Epoch 200/300
1/1 [==============================] - 0s 75ms/sample - loss: 13.4407
Epoch 201/300
1/1 [==============================] - 0s 70ms/sample - loss: 13.4407
Epoch 202/300
1/1 [==============================] - 0s 72ms/sample - loss: 13.4407
Epoch 203/300
1/1 [==============================] - 0s 72ms/sample - loss: 13.4407
Epoch 204/300
1/1 [==============================] - 0s 64ms/sample - loss: 13.4407
Epoch 205/300
1/1 [==============================] - 0s 67ms/sample - loss: 13.4407
Epoch 206/300
1/1 [==============================] - 0s 81ms/sample - loss: 13.4407
Epoch 207/300
1/1 [==============================] - 0s 69ms/sample - loss: 13.4407
Epoch 208/300
1/1 [==============================] - 0s 73ms/sample - loss: 13.4407
Epoch 209/300
1/1 [==============================] - 0s 65ms/sample - loss: 

Epoch 296/300
1/1 [==============================] - 0s 67ms/sample - loss: 13.4407
Epoch 297/300
1/1 [==============================] - 0s 65ms/sample - loss: 13.4407
Epoch 298/300
1/1 [==============================] - 0s 70ms/sample - loss: 13.4407
Epoch 299/300
1/1 [==============================] - 0s 64ms/sample - loss: 13.4407
Epoch 300/300
1/1 [==============================] - 0s 79ms/sample - loss: 13.4407


In [8]:
yhat = model.predict(valida, verbose=1)
print(yhat)

1/1 [==============================] - 0s 330ms/sample
[[[nan nan nan ... nan nan nan]
  [nan nan nan ... nan nan nan]
  [nan nan nan ... nan nan nan]
  ...
  [nan nan nan ... nan nan nan]
  [nan nan nan ... nan nan nan]
  [nan nan nan ... nan nan nan]]]
